In [1]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import requests
from dotenv import load_dotenv
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer

c:\Users\Himanshu\Desktop\wtc\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
## Reading documents from the file system
from pathlib import Path

directory_path = (Path.cwd().parent / "Web_Scrapper").resolve()
print("Loading from:", directory_path)

loader = DirectoryLoader(
    path=str(directory_path),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
    recursive=True,
 )

docs = loader.load()
print(len(docs), "documents loaded.")

Loading from: C:\Users\Himanshu\Desktop\wtc\wtc-round-2-group-5-syntax-error\Web_Scrapper


100%|██████████| 2/2 [00:00<00:00, 17.07it/s]

2 documents loaded.


In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250)
chunk = text_splitter.split_documents(docs)
print(len(chunk), "chunks created.")

1909 chunks created.


In [8]:
type(chunk[2])

langchain_core.documents.base.Document

In [9]:
chunk[2]

Document(metadata={'source': 'C:\\Users\\Himanshu\\Desktop\\wtc\\wtc-round-2-group-5-syntax-error\\Web_Scrapper\\gehu-homepage.txt'}, page_content='All three campuses strive to create a nurturing environment that helps students grow and thrive in all aspects of their lives. Our campuses are home to a diverse student population who come from all over the world, contributing to a vibrant and inclusive learning community.\nAt GEHU it is our commitment to provide high-quality education and research opportunities that contribute to knowledge creation and innovation, not just in our region, but on an international scale. Our faculty members are experts in their respective fields and are dedicated to inspiring and mentoring the next generation of leaders.\nGEHU Dehradun, Bhimtal, and Haldwani campuses provide a range of co-curricular activities, cultural events, and social initiatives that help students to develop their skills, passions, and interests outside of the classroom.')

In [10]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
text = [doc.page_content for doc in chunk]
embeddings = model.encode(
    text,
    batch_size=32,
    show_progress_bar=True
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1824.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 60/60 [01:12<00:00,  1.21s/it]


In [11]:
print(len(embeddings), "embeddings created.")
print("Dimension of each embedding:", len(embeddings[0]))
print("Sample embedding:", embeddings[0][:5])
print(len(chunk), "chunks and", len(embeddings), "embeddings should match.")

1909 embeddings created.
Dimension of each embedding: 384
Sample embedding: [ 0.01955767  0.07343878 -0.01102463 -0.08509269 -0.00771436]
1909 chunks and 1909 embeddings should match.


In [12]:
client = MongoClient(os.getenv("MONGO_URI"))

db = client["campus_ai"]
collection = db["academic_vector"]
vectors = []

for i, embedding in enumerate(embeddings):
    vectors.append({
        "id": f"chunk-{i}",
        "values": embedding.tolist(),
        "metadata": {
            "text": chunk[i].page_content,
            "source": chunk[i].metadata.get("source"),
            "page": chunk[i].metadata.get("page")
        }
    })
collection.insert_many(vectors)

InsertManyResult([ObjectId('69c6d1e582f3a0a906873249'), ObjectId('69c6d1e582f3a0a90687324a'), ObjectId('69c6d1e582f3a0a90687324b'), ObjectId('69c6d1e582f3a0a90687324c'), ObjectId('69c6d1e582f3a0a90687324d'), ObjectId('69c6d1e582f3a0a90687324e'), ObjectId('69c6d1e582f3a0a90687324f'), ObjectId('69c6d1e582f3a0a906873250'), ObjectId('69c6d1e582f3a0a906873251'), ObjectId('69c6d1e582f3a0a906873252'), ObjectId('69c6d1e582f3a0a906873253'), ObjectId('69c6d1e582f3a0a906873254'), ObjectId('69c6d1e582f3a0a906873255'), ObjectId('69c6d1e582f3a0a906873256'), ObjectId('69c6d1e582f3a0a906873257'), ObjectId('69c6d1e582f3a0a906873258'), ObjectId('69c6d1e582f3a0a906873259'), ObjectId('69c6d1e582f3a0a90687325a'), ObjectId('69c6d1e582f3a0a90687325b'), ObjectId('69c6d1e582f3a0a90687325c'), ObjectId('69c6d1e582f3a0a90687325d'), ObjectId('69c6d1e582f3a0a90687325e'), ObjectId('69c6d1e582f3a0a90687325f'), ObjectId('69c6d1e582f3a0a906873260'), ObjectId('69c6d1e582f3a0a906873261'), ObjectId('69c6d1e582f3a0a9068732